# SolarTech Lab telemetry — per-day viewer

Prints the measured photovoltaic telemetry for a chosen set of days, as intensity against time of
day. These are the values the control agent of Section 11 would be reading.

Set `DAYS` to a list of integers. Each is a **day of year** in 2017, `1`–`365`.

The notebook also reports, for each requested day, the longest run of consecutive valid samples —
which is what decides whether that day can be fed to the decomposition notebook, since the model
needs `CANON_LEN = CTX + PRED = 544` contiguous samples and `PV_Power` is `NaN` outside daylight.

## 0, Configuration

In [ ]:
# ------------------------------------------------------------------------------------- #
#  CONFIGURATION
# ------------------------------------------------------------------------------------- #
DAYS = [186, 158, 212, 15]        # day of year, 1-365. Any length; one panel per day.

CHANNEL   = "PV_Power"            # PV_Power | T_air | G_h | G_tilt | W_s | W_d
CSV_PATH  = "../data/dataset/Dataset-SolarTechLab.csv"

STACKED      = True               # one cumulative figure, the same panels one under another
OVERLAY      = True                # one extra figure with every requested day superimposed
SAVE_FIGURES = True
OUT_DIR      = "_run/solar"

# The tokenisation geometry the cpp/cps table below is written for.
REF_P, REF_S = 16, 16


## 1, Loading

The file is semicolon-separated with a `dd-MMM-yyyy HH:MM:SS` timestamp. Two things are cleaned on
the way in and both are properties of the recording, not of this notebook:

* `PV_Power` is `NaN` outside daylight. That is structural — the plant is not producing — so those
  samples are left as `NaN` rather than filled with zero, which would invent a signal.
* the weather channels carry `-999999` as a missing-value sentinel, which is mapped to `NaN`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

FS, CTX, PRED = 512, 480, 64          # the design's convention; see section 4 below
CANON_LEN = CTX + PRED

RAW = pd.read_csv(CSV_PATH, sep=";", parse_dates=["Time"], dayfirst=True)
RAW = RAW.replace(-999999.0, np.nan)
RAW["doy"] = RAW["Time"].dt.dayofyear
RAW["tod"] = RAW["Time"].dt.hour + RAW["Time"].dt.minute / 60.0

assert CHANNEL in RAW.columns, f"{CHANNEL} not in {list(RAW.columns)}"
UNITS = {"PV_Power": "W", "T_air": "°C", "G_h": "W/m²",
         "G_tilt": "W/m²", "W_s": "m/s", "W_d": "°"}
UNIT = UNITS.get(CHANNEL, "")

step = RAW["Time"].diff().mode()[0]
print(f"{len(RAW):,} rows   {RAW.Time.min().date()} → {RAW.Time.max().date()}   cadence {step}")
print(f"{CHANNEL}: {RAW[CHANNEL].notna().sum():,} valid "
      f"({100 * RAW[CHANNEL].notna().mean():.1f}%), "
      f"range {RAW[CHANNEL].min():.3g} – {RAW[CHANNEL].max():.3g} {UNIT}")


## 2, What each requested day contains

`run` is the longest block of consecutive valid samples. A day is usable by the decomposition
notebook only when `run` reaches 544.

In [ ]:
def longest_valid_run(values):
    """(length, start_index) of the longest consecutive non-NaN block."""
    ok = np.asarray(pd.notna(values))
    best = best_at = cur = 0
    start = 0
    for i, v in enumerate(ok):
        if v:
            if cur == 0:
                start = i
            cur += 1
            if cur > best:
                best, best_at = cur, start
        else:
            cur = 0
    return best, best_at


def day_frame(doy):
    d = RAW[RAW.doy == doy].sort_values("Time").reset_index(drop=True)
    if d.empty:
        raise ValueError(f"day {doy} is not in the file")
    return d


rows = []
for doy in DAYS:
    d = day_frame(doy)
    run, at = longest_valid_run(d[CHANNEL])
    rows.append({
        "doy": doy,
        "date": d.Time.iloc[0].date(),
        "samples": len(d),
        "valid": int(d[CHANNEL].notna().sum()),
        "run": run,
        "run_from": str(d.Time.iloc[at].time())[:5] if run else "-",
        "peak": round(float(d[CHANNEL].max()), 1) if d[CHANNEL].notna().any() else np.nan,
        "usable_544": "yes" if run >= CANON_LEN else "no",
    })
SUMMARY = pd.DataFrame(rows)
print(SUMMARY.to_string(index=False))


### Every day in the file, ranked by usable run

Run this when you want to choose days rather than check them. It is the list the decomposition
notebook draws from.

In [ ]:
ALL = []
for doy, d in RAW.groupby("doy"):
    run, at = longest_valid_run(d[CHANNEL])
    ALL.append({"doy": int(doy), "date": d.Time.iloc[0].date(), "run": run,
                "peak": round(float(d[CHANNEL].max()), 1) if d[CHANNEL].notna().any() else np.nan})
ALL = pd.DataFrame(ALL).sort_values("run", ascending=False).reset_index(drop=True)

n_ok = int((ALL.run >= CANON_LEN).sum())
print(f"{n_ok} of {len(ALL)} days carry {CANON_LEN} contiguous valid {CHANNEL} samples\n")
print(ALL.head(20).to_string(index=False))


## 3, The telemetry

One **separate figure per requested day**: every valid sample as a point, positioned by time of day
and coloured by its own value, so a thin arc is a clear day and a ragged one is a day with cloud
transients. Each figure carries its own colour scale and is written to its own file.

**The grey band is the longest run of consecutive valid samples in that day.** It is not a
measurement and not a confidence interval: it is a bookkeeping mark. `PV_Power` is `NaN` at night
and drops out whenever the logger misses a minute, so a day is a set of fragments rather than one
series. A model can only be handed an unbroken stretch, and the design needs
`CANON_LEN = CTX + PRED = 544` samples of it. The band shows where in the day that longest unbroken
stretch sits and, through its label, how long it is. Points falling outside it are perfectly valid
readings; they simply belong to a shorter fragment. Where the band is shorter than $544$ the day is
unusable for the decomposition notebook, and the legend says so.

In [ ]:
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

SPAN_KW = dict(color="0.55", alpha=0.14, zorder=0)


def run_label(run):
    """The legend text for the grey band."""
    if not run:
        return "no valid sample in this day"
    verdict = f"usable, >= {CANON_LEN}" if run >= CANON_LEN else f"too short, < {CANON_LEN}"
    return f"longest contiguous valid run: {run} samples ({verdict})"


def draw_day(ax, doy, norm=None, annotate_run=True):
    """Draw one day onto ax. Returns (scatter, run length)."""
    d = day_frame(doy)
    run, at = longest_valid_run(d[CHANNEL])
    ok = d[d[CHANNEL].notna()]

    sc = ax.scatter(ok.tod, ok[CHANNEL], c=ok[CHANNEL], cmap="viridis",
                    norm=norm, s=6, linewidths=0, alpha=0.9,
                    label=f"valid {CHANNEL} sample (colour = value)")
    if run:
        ax.axvspan(d.tod.iloc[at], d.tod.iloc[at + run - 1], **SPAN_KW)

    ttl = (f"day {doy}  ({d.Time.iloc[0].date()})   peak {d[CHANNEL].max():.0f} {UNIT}   "
           f"valid {d[CHANNEL].notna().sum()}/1440")
    if annotate_run:
        ttl += f"   longest run {run}"
    ax.set_title(ttl, fontsize=9)
    ax.set_xlabel("time of day [h]")
    ax.set_ylabel(f"{CHANNEL} [{UNIT}]")
    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 3))
    ax.grid(alpha=0.25, lw=0.5)
    return sc, run


def figure_for_day(doy):
    """One standalone figure for one day, with its own legend and colour scale."""
    fig, ax = plt.subplots(figsize=(11, 4.4))
    sc, run = draw_day(ax, doy, annotate_run=False)
    fig.colorbar(sc, ax=ax, pad=0.015, label=f"{CHANNEL} [{UNIT}]")
    ax.set_title(f"SolarTech Lab — {CHANNEL},  " + ax.get_title(), fontsize=10)
    ax.legend(handles=[Line2D([], [], marker="o", ls="", ms=5, color="#3a7d8c",
                              label=f"valid {CHANNEL} sample (colour = value)"),
                       Patch(facecolor="0.55", alpha=0.30, label=run_label(run))],
              loc="upper left", fontsize=8, frameon=True, framealpha=0.85)
    fig.tight_layout()
    return fig


if SAVE_FIGURES:
    Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

for doy in DAYS:
    fig = figure_for_day(doy)
    if SAVE_FIGURES:
        p = Path(OUT_DIR) / f"solar_{CHANNEL}_day{doy:03d}.png"
        fig.savefig(p, dpi=160, bbox_inches="tight")
        print("wrote", p)
    plt.show()


### The cumulative figure

The same panels, one under another, on a **single colour scale** shared by every day so that panel
heights and colours are comparable, and with a **single legend** for the whole figure. Each panel
title carries its own longest-run length.

In [ ]:
if STACKED and len(DAYS) >= 1:
    import matplotlib.colors as mcolors

    vals = pd.concat([day_frame(doy)[CHANNEL].dropna() for doy in DAYS])
    NORM = mcolors.Normalize(vmin=float(vals.min()), vmax=float(vals.max()))

    n = len(DAYS)
    fig, axes = plt.subplots(n, 1, figsize=(11, 2.9 * n), squeeze=False, sharex=True)
    runs = []
    for ax, doy in zip(axes[:, 0], DAYS):
        sc, run = draw_day(ax, doy, norm=NORM, annotate_run=True)
        runs.append(run)
    for ax in axes[:-1, 0]:
        ax.set_xlabel("")

    fig.subplots_adjust(hspace=0.42, top=0.90, bottom=0.09)
    fig.colorbar(sc, ax=axes[:, 0].tolist(), pad=0.015, label=f"{CHANNEL} [{UNIT}]")

    n_ok_req = sum(r >= CANON_LEN for r in runs)
    fig.legend(handles=[Line2D([], [], marker="o", ls="", ms=5, color="#3a7d8c",
                               label=f"valid {CHANNEL} sample (colour = value, shared scale)"),
                        Patch(facecolor="0.55", alpha=0.30,
                              label=f"longest contiguous valid run of that day "
                                    f"(length in each title; {n_ok_req}/{n} reach {CANON_LEN})")],
               loc="upper center", bbox_to_anchor=(0.44, 0.965),
               ncol=2, fontsize=8, frameon=False)
    fig.suptitle(f"SolarTech Lab — {CHANNEL}, requested days stacked", y=0.995, fontsize=11)

    if SAVE_FIGURES:
        p = Path(OUT_DIR) / f"solar_{CHANNEL}_stacked_{'-'.join(map(str, DAYS))}.png"
        fig.savefig(p, dpi=160, bbox_inches="tight")
        print("wrote", p)
    plt.show()


In [ ]:
if OVERLAY and len(DAYS) > 1:
    fig, ax = plt.subplots(figsize=(11, 4))
    for doy in DAYS:
        d = day_frame(doy)
        ok = d[d[CHANNEL].notna()]
        ax.plot(ok.tod, ok[CHANNEL], lw=0.9, alpha=0.85,
                label=f"day {doy} ({d.Time.iloc[0].date()})")
    ax.set_xlabel("time of day [h]"); ax.set_ylabel(f"{CHANNEL} [{UNIT}]")
    ax.set_xlim(0, 24); ax.set_xticks(range(0, 25, 3))
    ax.grid(alpha=0.25, lw=0.5); ax.legend(fontsize=8, frameon=False)
    ax.set_title(f"{CHANNEL}, requested days superimposed", fontsize=10)
    if SAVE_FIGURES:
        p = Path(OUT_DIR) / f"solar_{CHANNEL}_overlay.png"
        fig.savefig(p, dpi=160, bbox_inches="tight"); print("wrote", p)
    plt.show()


## 4, Reading these minutes in the report's units

The synthetic design is written at $f_s = 512$ Hz and this series is one sample per minute. The two
are reconciled **without resampling**, because the quantity the report carries between them is
dimensionless.

A tokeniser sees a sequence of samples and has no notion of seconds. A component whose period is
$T$ **samples** completes $P/T$ cycles inside one patch and $S/T$ inside one stride, whatever the
wall-clock rate:

$$\mathrm{cpp} = \frac{P}{T}, \qquad \mathrm{cps} = \frac{S}{T}.$$

So a solar window is fed to the model as-is, and its sample index is read as if sampled at
$f_s = 512$ Hz. A component of period $T$ minutes then appears at $f = 512/T$ Hz on the notebook's
axis and at the same $\mathrm{cpp}$ it would have had in the synthetic study. The lock condition —
a whole number of cycles per patch or per stride — is therefore **the same predicate in both
worlds**, which is what makes the synthetic result transferable at all.

The table below is that mapping for the reference geometry.

In [ ]:
periods_min = [4, 8, 12, 16, 20, 24, 32, 48, 60, 96, 120, 240, 480]
tab = pd.DataFrame({
    "period [min]": periods_min,
    "f at fs=512 [Hz]": [round(FS / T, 2) for T in periods_min],
    "cpp (P=%d)" % REF_P: [round(REF_P / T, 4) for T in periods_min],
    "cps (S=%d)" % REF_S: [round(REF_S / T, 4) for T in periods_min],
})
tab["patch lock"] = ["yes" if abs(v - round(v)) < 1e-9 and v >= 1 else ""
                     for v in (REF_P / np.array(periods_min))]
tab["stride lock"] = ["yes" if abs(v - round(v)) < 1e-9 and v >= 1 else ""
                      for v in (REF_S / np.array(periods_min))]
print(f"reference geometry P={REF_P}, S={REF_S}\n")
print(tab.to_string(index=False))
print(f"\nA candidate site is a whole number of cycles per patch or per stride.")
print(f"At P={REF_P} that is a period of {REF_P} minutes (cpp=1), {REF_P/2:g} minutes (cpp=2), "
      f"{REF_P/3:.2f} minutes (cpp=3), and so on.")


## 5, What this notebook is for, and what it is not

It shows the telemetry the agent of Section 11 would read, and it decides which days can be handed
to the decomposition notebook. It performs no test: the photovoltaic series is the application
domain of this study and not an experimental corpus, and no hypothesis is evaluated on it.